In [0]:
dbutils.widgets.text(name="env", defaultValue="", label="Enter the envoronment in lower case")
db = dbutils.widgets.get("env")

##Read Silver Traffic Table

In [0]:
def read_SilverTrafficTable(environment):
    print("Reading Silver Traffic table", end = "")
    df_SilverTraffic = (spark.reaStream.table(f'`{environment}_catalog`.`silver`.`silver_traffic`'))
    print(f"Reading {environment}_catalog.silver.silver_traffic")
    return df_SilverTraffic

##Read Silver Roads Table

In [0]:
def read_SilverRoadsTable(environment):
    print("Reading Silver Traffic table", end = "")
    df_SilverRoads = (spark.reaStream.table(f'`{environment}_catalog`.`silver`.`silver_roads`'))
    print(f"Reading {environment}_catalog.silver.silver_roads")
    return df_SilverRoads

##Creating Vehicle intensity column

In [0]:
def create_vehicleintensity(df):
    from pyspark.sql.functions import col
    print("Creating vehicle intensity", end = "")
    df_vehicle_intensity  = df.withColumn("vehicleintensity", col("Motor_Vehicles_Count")/col("Link_length+km"))
    print("Created vehicle intensity")
    return df_vehicle_intensity


## Creating load time column

In [0]:
def create_Load_time(df):
    print("Creating Load time", end = "")
    df_load_time = df.withColumn("Load_time", current_timestamp())
    print("Created Load time")
    return df_load_time


## Writing data to Gold Traffic

In [0]:
def writ_traffic_Goldtable(streamingDF, environment):
    print("Writing Gold table", end = "")
    write_gold_roads = (streamingDF.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation",checkpoint+ "/GoldRoadsLoad/checkpoint/")
        .queryName("GoldRoadsLoad")
        .trigger(availableNow=True)
        .toTable(f"`{environment}_catalog`.`Gold`.`GoldRoadsLoad`"))

    write_gold_roads.awaitTermination()
    print(f"Written `{environment}_catalog`.`Gold`.`GoldRoadsLoad` success")

In [0]:
## Reading from silver tables
df_SilverTraffic = read_SilverTrafficTable(env)
df_SilverRoads = read_SilverRoadsTable(env)

##Transformations
df_vehicle_intensity = create_vehicleintensity(df_SilverTraffic)
df_load_time = create_Load_time(df_vehicle_intensity)
df_finalRoads = create_Load_time(df_SilverRoads)

##writing to gold tables
writ_traffic_Goldtable(df_load_time, env)
write_Roads_Goldtable(df_finalRoads, env)